## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        # 输入维度 784 (28x28), 隐藏层 256 个神经元
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        
        # 输出层 10 个神经元（对应0-9数字分类）
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        # 展平输入 [batch, 28, 28] => [batch, 784]
        x = tf.reshape(x, [-1, 784])
        
        # 第一层：全连接 + ReLU激活
        h = tf.nn.relu(x @ self.W1 + self.b1)
        
        # 输出层：全连接（不激活）
        logits = h @ self.W2 + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.6513603 ; accuracy 0.1458
epoch 1 : loss 2.5955281 ; accuracy 0.1508
epoch 2 : loss 2.5469594 ; accuracy 0.1552
epoch 3 : loss 2.5037477 ; accuracy 0.15951666
epoch 4 : loss 2.464686 ; accuracy 0.16303334
epoch 5 : loss 2.4289458 ; accuracy 0.16726667
epoch 6 : loss 2.3959405 ; accuracy 0.17246667
epoch 7 : loss 2.3652296 ; accuracy 0.17791666
epoch 8 : loss 2.3364735 ; accuracy 0.18355
epoch 9 : loss 2.3093984 ; accuracy 0.19015
epoch 10 : loss 2.2837818 ; accuracy 0.19711667
epoch 11 : loss 2.2594411 ; accuracy 0.20473333
epoch 12 : loss 2.236224 ; accuracy 0.21255
epoch 13 : loss 2.213998 ; accuracy 0.2213
epoch 14 : loss 2.1926556 ; accuracy 0.22988333
epoch 15 : loss 2.172103 ; accuracy 0.23836666
epoch 16 : loss 2.152257 ; accuracy 0.24616666
epoch 17 : loss 2.1330497 ; accuracy 0.25461668
epoch 18 : loss 2.1144185 ; accuracy 0.26283333
epoch 19 : loss 2.0963106 ; accuracy 0.2711
epoch 20 : loss 2.0786803 ; accuracy 0.27928334
epoch 21 : loss 2.061488 ; accuracy 